# Dynamic Pricing Simulation for Ride-Hailing Platforms

This notebook demonstrates an end-to-end pipeline for:
1. **Data Generation** — Realistic synthetic ride-hailing data
2. **Demand Prediction** — Linear Regression, Random Forest, XGBoost
3. **Dynamic Pricing** — Surge multiplier based on demand/supply
4. **Simulation** — Revenue, rides served, summary statistics
5. **Visualization** — Demand vs supply, surge over time, price distribution

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src.data_processing import generate_synthetic_data, aggregate_to_hourly, prepare_ml_features
from src.demand_model import evaluate_models, get_best_model
from src.pricing_model import apply_pricing, PricingConfig
from src.visualization import generate_all_plots

## 2. Load and Prepare Data

In [ ]:
# Generate synthetic ride-hailing data (7 days)
df_raw = generate_synthetic_data(start_date='2024-01-01', end_date='2024-01-08', seed=42)
df = aggregate_to_hourly(df_raw)

print(f"Data shape: {df.shape}")
df.head(10)

## 3. Train Demand Prediction Models

In [ ]:
X, y = prepare_ml_features(df)
models, metrics = evaluate_models(X, y)
best_model_name = get_best_model(metrics, criterion='RMSE')

print("Model Performance (MAE, RMSE, R²):")
for name, m in metrics.items():
    print(f"  {name}: MAE={m['MAE']:.2f}, RMSE={m['RMSE']:.2f}, R²={m['R2']:.3f}")
print(f"\nBest model: {best_model_name}")

df['demand_predicted'] = models[best_model_name].predict(X)

## 4. Apply Dynamic Pricing

In [ ]:
config = PricingConfig(min_multiplier=1.0, max_multiplier=3.0)
df = apply_pricing(df, config, demand_col='demand_predicted')

df['rides_served'] = df[['demand', 'driver_supply']].min(axis=1)
df['revenue'] = df['rides_served'] * df['price']

df[['timestamp', 'demand', 'driver_supply', 'surge_multiplier', 'price', 'revenue']].head(10)

## 5. Summary Statistics

In [ ]:
total_revenue = df['revenue'].sum()
avg_price = df['price'].mean()
demand_served_ratio = df['rides_served'].sum() / df['demand'].sum()

print(f"Total Revenue:        ${total_revenue:,.2f}")
print(f"Average Price:        ${avg_price:.2f}")
print(f"Demand Served Ratio:  {demand_served_ratio:.2%}")

## 6. Visualizations

In [ ]:
figures_dir = Path.cwd().parent / 'figures'
generate_all_plots(df, figures_dir)
print(f"Figures saved to {figures_dir}")